In [ ]:
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
import operator
from langgraph.graph import StateGraph , START, END


llm = ChatOpenAI(
    model="gpt-4.1-mini",  # Your Azure deployment name
    base_url="https://openai-rg-nadeem.openai.azure.com/openai/v1",
    api_key=    )


class State(TypedDict):
    question: str
    answer: str
    explanation: str
    summary: str
    history: Annotated[list[str], operator.add]


def chat_node(state: State) -> State:
   result = llm.invoke(state['question'])
   answer = result.content
   state['answer'] = answer
   return state

def expl_node(state: State):
    prompt = f"This is the question: {state['question']} and this is the answer {state['answer']}... you have to explain this"
    result = llm.invoke(prompt)
    state['explanation']= result.content
    return {"explanation":state['explanation']}

def sum_node(state: State):
    prompt = f"This is the question: {state['question']} and this is the answer {state['answer']}... you have to summarize this"
    result = llm.invoke(prompt)
    state['summary']= result.content
    return {"summary":state['summary']}


graph = StateGraph(State)


graph.add_node('chat_node', chat_node)
graph.add_node('expl_node', expl_node)
graph.add_node('summary_node', sum_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', 'summary_node')
graph.add_edge('chat_node', 'expl_node')
graph.add_edge('expl_node', END)
graph.add_edge('summary_node', END)

workflow = graph.compile()
initial_state = {"question": "whats your name", "history":["hello", "hi"]}
result = workflow.invoke(initial_state)
print(result)


{'question': 'whats your name', 'answer': "I'm ChatGPT! How can I assist you today?", 'explanation': 'The question "What\'s your name?" is asking for the identity or designation of the person (or entity) being addressed. In response, saying "I\'m ChatGPT! How can I assist you today?" serves two purposes:\n\n1. **Providing the Name:** The answer clearly states the name or identity, which is "ChatGPT."\n2. **Offering Assistance:** By adding "How can I assist you today?" the responder signals readiness to help with any questions or tasks, making the interaction more friendly and engaging.\n\nIn summary, the answer correctly identifies itself while also inviting further interaction.', 'summary': "The answer is: I'm ChatGPT. How can I help you?", 'history': ['hello', 'hi', 'hello', 'hi']}


whatever we return from the node function it automatically gets the value of the attribute of the object that we created